In [1]:
import pandas as pd
import numpy as np
import os
import torch
import librosa
import noisereduce as nr
from tqdm import tqdm
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
import warnings
import pickle
from collections import defaultdict

warnings.filterwarnings('ignore')

# =============================================================================
# 설정
# =============================================================================
BASE_PATH = r"D:\depression_dataset(DAIC-WOZ)"
EXCEPTION_NUMBER = [
    '451', '458', '480']
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 하이퍼파라미터
MAX_UTTERANCE_DURATION = 15.0   # 최대 발화 길이 (초) - 이상 분할
MIN_UTTERANCE_DURATION = 0.5    # 최소 발화 길이 (초) - 이하 제거
SR = 16000

# Question Type 매핑 (단순화)
Q_TYPE_MAPPING = {
    'casual': 0,      # small talk, preference, open-ended encouragement
    'background': 1,  # daily habits, social, self-perception
    'emotional': 2,   # emotion / mood
    'clinical': 3,    # depression symptoms direct
    'other': 4
}

# 원본 → 단순화 매핑
Q_TYPE_SIMPLIFICATION = {
    'small talk': 'casual',
    'preference': 'casual',
    'open-ended encouragement': 'casual',
    
    'daily habits / lifestyle': 'background',
    'social / family / relationship': 'background',
    'self-perception / personality': 'background',
    
    'emotion / mood': 'emotional',
    
    'depression symptoms direct': 'clinical',
    
    'other': 'other'
}

print(f"⏳ WavLM 모델 로딩 중... (Device: {DEVICE})")
try:
    # Safetensors 우선 사용
    wavlm_feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
    wavlm_model = WavLMModel.from_pretrained(
        "microsoft/wavlm-base-plus",
        use_safetensors=True  # safetensors 명시적 사용
    ).to(DEVICE)
    wavlm_model.eval()
    print("✅ WavLM 모델 로드 완료! (safetensors)")
except Exception as e:
    print(f"⚠️  Safetensors 로드 실패, 재시도 중...")
    # 다운로드 강제 재시도
    wavlm_feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
        "microsoft/wavlm-base-plus",
        force_download=False
    )
    wavlm_model = WavLMModel.from_pretrained(
        "microsoft/wavlm-base-plus",
        use_safetensors=True,
        trust_remote_code=False
    ).to(DEVICE)
    wavlm_model.eval()
    print("✅ WavLM 모델 로드 완료!")


# =============================================================================
# 텍스트 특징 추출 (TTR만)
# =============================================================================
def get_ttr(text):
    """Type-Token Ratio 계산"""
    if not text or len(text.strip()) == 0:
        return 0.0
    tokens = text.lower().split()
    if len(tokens) == 0:
        return 0.0
    return len(set(tokens)) / len(tokens)


def extract_linguistic_features(text):
    """
    추가 언어학적 특징 추출
    
    Returns:
        dict: {
            'word_count': int,
            'avg_word_length': float,
            'sentence_count': int,
            'negation_count': int,
            'first_person_count': int
        }
    """
    if not text or len(text.strip()) == 0:
        return {
            'word_count': 0,
            'avg_word_length': 0.0,
            'sentence_count': 0,
            'negation_count': 0,
            'first_person_count': 0
        }
    
    text_lower = text.lower()
    words = text_lower.split()
    
    # 단어 수
    word_count = len(words)
    
    # 평균 단어 길이
    avg_word_length = np.mean([len(w) for w in words]) if words else 0.0
    
    # 문장 수 (간단한 추정)
    sentence_count = max(1, text.count('.') + text.count('!') + text.count('?'))
    
    # 부정어 카운트
    negation_words = ['no', 'not', 'never', "n't", 'nothing', 'nobody', 'nowhere', 
                     'neither', 'hardly', 'barely', 'scarcely', "don't", "didn't", 
                     "won't", "wouldn't", "can't", "couldn't", "shouldn't"]
    negation_count = sum(1 for word in words if word in negation_words)
    
    # 1인칭 대명사 카운트 (우울증 환자에서 높음)
    first_person = ['i', 'me', 'my', 'mine', 'myself']
    first_person_count = sum(1 for word in words if word in first_person)
    
    return {
        'word_count': word_count,
        'avg_word_length': avg_word_length,
        'sentence_count': sentence_count,
        'negation_count': negation_count,
        'first_person_count': first_person_count
    }


# =============================================================================
# WavLM 특징 추출
# =============================================================================
def extract_wavlm_features(audio_waveform, sampling_rate):
    """WavLM 특징 추출"""
    try:
        if len(audio_waveform) < SR * 0.3:  # 0.3초 미만은 너무 짧음
            return None
        
        # Feature extraction
        wavlm_inputs = wavlm_feature_extractor(
            audio_waveform, 
            sampling_rate=sampling_rate, 
            return_tensors="pt", 
            padding=True
        )
        wavlm_input_values = wavlm_inputs.input_values.to(DEVICE)
        
        with torch.no_grad():
            wavlm_outputs = wavlm_model(wavlm_input_values)
            wavlm_hidden_states = wavlm_outputs.last_hidden_state  # [1, time_steps, 768]
        
        # 평균 풀링
        wavlm_embedding = torch.mean(wavlm_hidden_states, dim=1).squeeze().cpu().numpy()
        
        # 안전성 체크
        if not np.isfinite(wavlm_embedding).all():
            return None
        
        return wavlm_embedding
    except Exception as e:
        return None


# =============================================================================
# 대화 전처리 클래스
# =============================================================================
class UtterancePreprocessor:
    def __init__(self, base_path):
        self.base_path = base_path
    
    def normalize_question_type(self, q_type):
        """질문 유형 정규화 및 단순화"""
        q_type = q_type.lower().strip()
        q_type = ' '.join(q_type.split())  # 공백 정규화
        q_type = q_type.replace('/', ' / ')
        q_type = ' '.join(q_type.split())
        
        # 단순화
        if q_type in Q_TYPE_SIMPLIFICATION:
            return Q_TYPE_SIMPLIFICATION[q_type]
        return 'other'
    
    def process_transcript(self, pid):
        """CSV에서 발화 단위로 추출"""
        transcript_path = os.path.join(self.base_path, f"{pid}_P", f"{pid}_cleaned_transcript.csv")
        
        try:
            df = pd.read_csv(transcript_path, sep='\t')
            if df.shape[1] < 2:
                df = pd.read_csv(transcript_path, sep=',')
        except:
            return []
        
        # 컬럼명 정규화
        df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
        
        # question_label 처리
        if 'question_label' not in df.columns:
            df['question_label'] = 'other'
        df['question_label'] = df['question_label'].fillna('other')
        df['question_label'] = df['question_label'].replace('', 'other')
        
        # 발화 추출
        utterances = self._extract_utterances(df, pid)
        
        # 긴 발화 분할
        final_utterances = self._split_long_utterances(utterances)
        
        return final_utterances
    
    def _extract_utterances(self, df, pid):
        """발화 단위로 추출 (합치지 않음)"""
        utterances = []
        current_q_type = None
        first_ellie_found = False
        
        for idx, row in df.iterrows():
            speaker = str(row['speaker']).strip().lower()
            q_label = str(row['question_label']).strip().lower()
            
            # Ellie 발화
            if 'ellie' in speaker:
                first_ellie_found = True
                
                # 질문 유형 업데이트
                q_label = self.normalize_question_type(q_label)
                if q_label != 'other':
                    current_q_type = q_label
                else:
                    current_q_type = 'other'
            
            # Participant 발화
            elif 'participant' in speaker:
                if not first_ellie_found:
                    continue  # 첫 Ellie 발화 전 무시
                
                if current_q_type is None:
                    continue
                
                text = str(row['value'])
                start_time = row['start_time']
                stop_time = row['stop_time']
                duration = stop_time - start_time
                
                # 너무 짧은 발화 제거
                if duration < MIN_UTTERANCE_DURATION:
                    continue
                
                utterances.append({
                    'pid': pid,
                    'q_type': current_q_type,
                    'text': text,
                    'start': start_time,
                    'end': stop_time,
                    'duration': duration
                })
        
        return utterances
    
    def _split_long_utterances(self, utterances):
        """긴 발화를 15초 단위로 분할"""
        final_utterances = []
        
        for utt in utterances:
            duration = utt['duration']
            
            if duration <= MAX_UTTERANCE_DURATION:
                final_utterances.append(utt)
            else:
                # 분할
                num_splits = int(np.ceil(duration / MAX_UTTERANCE_DURATION))
                split_duration = duration / num_splits
                
                for i in range(num_splits):
                    split_start = utt['start'] + i * split_duration
                    split_end = min(split_start + split_duration, utt['end'])
                    
                    final_utterances.append({
                        'pid': utt['pid'],
                        'q_type': utt['q_type'],
                        'text': utt['text'],  # 텍스트는 동일하게 유지
                        'start': split_start,
                        'end': split_end,
                        'duration': split_end - split_start
                    })
        
        return final_utterances


# =============================================================================
# 오디오 품질 검증
# =============================================================================
def check_audio_quality(y, sr):
    """오디오 품질 검사"""
    issues = []
    
    # 1. 무음 비율 체크 (80% 이상 무음이면 문제)
    energy = librosa.feature.rms(y=y)[0]
    silence_ratio = np.sum(energy < 0.01) / len(energy)
    if silence_ratio > 0.8:
        issues.append(f"무음 비율 높음: {silence_ratio:.2%}")
    
    # 2. 클리핑 체크 (진폭이 0.99 이상인 비율)
    clipping_ratio = np.sum(np.abs(y) > 0.99) / len(y)
    if clipping_ratio > 0.01:
        issues.append(f"클리핑 발생: {clipping_ratio:.2%}")
    
    # 3. 너무 작은 볼륨
    max_amplitude = np.max(np.abs(y))
    if max_amplitude < 0.01:
        issues.append(f"볼륨 너무 작음: {max_amplitude:.4f}")
    
    return issues


# =============================================================================
# 메인 파이프라인
# =============================================================================
def run_preprocessing_pipeline():
    """전체 전처리 파이프라인 실행"""
    # 메타데이터 로드
    meta = pd.read_csv(os.path.join(BASE_PATH, "metadataset.csv"))
    meta['Participant_ID'] = meta['Participant_ID'].astype(str)
    meta = meta[~meta['Participant_ID'].isin(EXCEPTION_NUMBER)].reset_index(drop=True)
    
    preprocessor = UtterancePreprocessor(BASE_PATH)
    
    # 참가자별 데이터 저장
    wavlm_dataset = {}
    
    print(f"\n{'='*70}")
    print(f"🚀 전처리 시작: {len(meta)}명")
    print(f"   - 오디오: WavLM (microsoft/wavlm-base-plus)")
    print(f"   - 텍스트: 언어학적 특징만 (TTR 등)")
    print(f"{'='*70}\n")
    
    stats = {
        'processed': 0,
        'failed': 0,
        'total_utterances': 0,
        'low_quality': 0,
        'q_type_counts': defaultdict(int)
    }
    
    quality_issues = []
    
    for idx, row in tqdm(meta.iterrows(), total=len(meta), desc="참가자 처리"):
        pid = str(row['Participant_ID'])
        label = int(row['Binary'])
        
        # 1. CSV에서 발화 추출
        utterances = preprocessor.process_transcript(pid)
        if not utterances:
            stats['failed'] += 1
            continue
        
        # 2. 오디오 로드 (전체 파일 한 번만)
        audio_path = os.path.join(BASE_PATH, f"{pid}_P", f"{pid}_AUDIO.wav")
        if not os.path.exists(audio_path):
            stats['failed'] += 1
            continue
        
        try:
            y_full, _ = librosa.load(audio_path, sr=SR)
            # 노이즈 제거
            y_full = nr.reduce_noise(y=y_full, sr=SR, stationary=True, prop_decrease=0.8)
        except Exception as e:
            stats['failed'] += 1
            continue
        
        # 3. 각 발화별 특징 추출
        processed_utterances = []
        
        for utt in utterances:
            # ==================== 오디오 특징 ====================
            # 오디오 추출
            start_sample = int(utt['start'] * SR)
            end_sample = int(utt['end'] * SR)
            
            if start_sample >= end_sample or end_sample > len(y_full):
                continue
            
            audio_segment = y_full[start_sample:end_sample]
            
            # 품질 검사
            issues = check_audio_quality(audio_segment, SR)
            if issues:
                quality_issues.append({
                    'pid': pid,
                    'duration': utt['duration'],
                    'issues': issues
                })
                stats['low_quality'] += 1
                # 심각한 문제 아니면 계속 진행
                if len(issues) > 2:  # 2개 이상 문제면 스킵
                    continue
            
            # WavLM 특징 추출
            wavlm_feat = extract_wavlm_features(audio_segment, SR)
            if wavlm_feat is None:
                continue
            
            # ==================== 텍스트 특징 ====================
            # 언어학적 특징
            linguistic_feats = extract_linguistic_features(utt['text'])
            
            # TTR 계산
            ttr = get_ttr(utt['text'])
            
            # Q-type ID 변환
            q_type_id = Q_TYPE_MAPPING.get(utt['q_type'], Q_TYPE_MAPPING['other'])
            
            # ==================== 저장 ====================
            utterance_data = {
                # 오디오 특징 (WavLM)
                'wavlm': wavlm_feat,  # [768]
                
                # 텍스트 특징 (언어학적 특징만)
                'ttr': ttr,
                'word_count': linguistic_feats['word_count'],
                'avg_word_length': linguistic_feats['avg_word_length'],
                'sentence_count': linguistic_feats['sentence_count'],
                'negation_count': linguistic_feats['negation_count'],
                'first_person_count': linguistic_feats['first_person_count'],
                
                # 메타 정보
                'q_type': utt['q_type'],
                'q_type_id': q_type_id,
                'duration': utt['duration'],
                'text': utt['text']
            }
            
            processed_utterances.append(utterance_data)
            
            stats['total_utterances'] += 1
            stats['q_type_counts'][utt['q_type']] += 1
        
        if not processed_utterances:
            stats['failed'] += 1
            continue
        
        # 4. 참가자 데이터 저장
        wavlm_dataset[pid] = {
            'label': label,
            'utterances': processed_utterances,
            'num_utterances': len(processed_utterances)
        }
        
        stats['processed'] += 1
    
    # 통계 출력
    print(f"\n{'='*70}")
    print(f"✅ 전처리 완료!")
    print(f"{'='*70}")
    print(f"처리 성공: {stats['processed']}명")
    print(f"처리 실패: {stats['failed']}명")
    print(f"총 발화: {stats['total_utterances']}개")
    print(f"품질 이슈: {stats['low_quality']}개 발화")
    
    print(f"\nQuestion Type 분포:")
    for q_type, count in sorted(stats['q_type_counts'].items(), key=lambda x: -x[1]):
        percentage = (count / stats['total_utterances']) * 100
        print(f"  {q_type:15s}: {count:5d}개 ({percentage:5.1f}%)")
    
    # 품질 이슈 샘플 출력
    if quality_issues:
        print(f"\n품질 이슈 샘플 (상위 10개):")
        for issue in quality_issues[:10]:
            print(f"  PID {issue['pid']}: {issue['duration']:.1f}초 - {', '.join(issue['issues'])}")
    
    return wavlm_dataset


# =============================================================================
# 실행 및 저장
# =============================================================================
if __name__ == "__main__":
    # 전처리 실행
    wavlm_dataset = run_preprocessing_pipeline()
    
    # 저장 (변수명 겹치지 않도록 wavlm 명시)
    output_path_wavlm = os.path.join(BASE_PATH, "preprocessed_utterance_dataset_wavlm_only.pkl")
    with open(output_path_wavlm, 'wb') as f:
        pickle.dump(wavlm_dataset, f)
    
    print(f"\n💾 데이터 저장 완료: {output_path_wavlm}")
    
    # 샘플 데이터 확인
    sample_pid = list(wavlm_dataset.keys())[0]
    sample = wavlm_dataset[sample_pid]
    
    print(f"\n{'='*70}")
    print(f"📋 샘플 데이터 구조 (PID: {sample_pid})")
    print(f"{'='*70}")
    print(f"Label: {sample['label']}")
    print(f"Num Utterances: {sample['num_utterances']}")
    print(f"\n첫 번째 발화:")
    utt = sample['utterances'][0]
    print(f"  - Q-type: {utt['q_type']} (ID: {utt['q_type_id']})")
    print(f"  - Duration: {utt['duration']:.2f}초")
    print(f"  - WavLM Shape: {utt['wavlm'].shape}")
    
    print(f"\n  텍스트 특징:")
    print(f"    - TTR: {utt['ttr']:.3f}")
    print(f"    - Word Count: {utt['word_count']}")
    print(f"    - Avg Word Length: {utt['avg_word_length']:.2f}")
    print(f"    - Sentence Count: {utt['sentence_count']}")
    print(f"    - Negation Count: {utt['negation_count']}")
    print(f"    - First Person Count: {utt['first_person_count']}")
    print(f"    - Text: {utt['text'][:80]}...")
    
    print(f"\n{'='*70}")
    print(f"✅ 모든 작업 완료!")
    print(f"{'='*70}")

c:\Users\Lenovo\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


⏳ WavLM 모델 로딩 중... (Device: cuda)


preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

✅ WavLM 모델 로드 완료! (safetensors)

🚀 전처리 시작: 186명
   - 오디오: WavLM (microsoft/wavlm-base-plus)
   - 텍스트: 언어학적 특징만 (TTR 등)



참가자 처리: 100%|██████████| 186/186 [37:08<00:00, 11.98s/it]



✅ 전처리 완료!
처리 성공: 186명
처리 실패: 0명
총 발화: 28580개
품질 이슈: 18248개 발화

Question Type 분포:
  background     : 11743개 ( 41.1%)
  casual         :  8313개 ( 29.1%)
  emotional      :  5607개 ( 19.6%)
  clinical       :  2917개 ( 10.2%)

품질 이슈 샘플 (상위 10개):
  PID 300: 3.1초 - 무음 비율 높음: 83.51%
  PID 300: 0.8초 - 무음 비율 높음: 100.00%
  PID 300: 1.5초 - 무음 비율 높음: 100.00%
  PID 300: 3.3초 - 무음 비율 높음: 100.00%
  PID 300: 0.6초 - 무음 비율 높음: 100.00%, 볼륨 너무 작음: 0.0087
  PID 300: 0.7초 - 무음 비율 높음: 100.00%
  PID 300: 0.8초 - 무음 비율 높음: 100.00%
  PID 300: 1.1초 - 무음 비율 높음: 94.12%
  PID 300: 1.5초 - 무음 비율 높음: 95.83%
  PID 300: 0.9초 - 무음 비율 높음: 100.00%

💾 데이터 저장 완료: D:\depression_dataset(DAIC-WOZ)\preprocessed_utterance_dataset_wavlm_only.pkl

📋 샘플 데이터 구조 (PID: 300)
Label: 0
Num Utterances: 83

첫 번째 발화:
  - Q-type: casual (ID: 0)
  - Duration: 0.85초
  - WavLM Shape: (768,)

  텍스트 특징:
    - TTR: 1.000
    - Word Count: 1
    - Avg Word Length: 4.00
    - Sentence Count: 1
    - Negation Count: 0
    - First Person Count: 0
    - 

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import pickle
import os
import math
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from optuna.trial import TrialState
import json

# =============================================================================
# 설정
# =============================================================================
BASE_PATH = r"D:\depression_dataset(DAIC-WOZ)"
WAVLM_PREPROCESSED_DATA_PATH = os.path.join(BASE_PATH, "preprocessed_utterance_dataset_wavlm_only.pkl")

# 모델 기본 설정
WAVLM_DIM = 768  # WavLM 임베딩 차원
Q_TYPE_VOCAB_SIZE = 5
Q_TYPE_EMBED_DIM = 32
TTR_ENHANCED_DIM = 6

# 학습 기본 설정 (Optuna로 조정될 항목 제외)
BATCH_SIZE = 8
NUM_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 15
GRADIENT_CLIP = 1.0

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Optuna 설정
N_TRIALS = 50
STUDY_NAME = "wavlm_ttr_enhanced_balanced_f1"
OPTUNA_EPOCHS = 20  # Optuna trial당 epoch 수


# =============================================================================
# Enhanced TTR Feature Extraction
# =============================================================================
def extract_enhanced_ttr_features(text):
    """TTR 관련 강화된 특징 추출"""
    if not text or len(text.strip()) == 0:
        return {
            'ttr': 0.0,
            'ttr_log': 0.0,
            'repetition_rate': 0.0,
            'unique_word_ratio': 0.0,
            'lexical_density': 0.0,
            'word_length_variance': 0.0
        }
    
    tokens = text.lower().split()
    
    if len(tokens) == 0:
        return {
            'ttr': 0.0,
            'ttr_log': 0.0,
            'repetition_rate': 0.0,
            'unique_word_ratio': 0.0,
            'lexical_density': 0.0,
            'word_length_variance': 0.0
        }
    
    unique_tokens = set(tokens)
    ttr = len(unique_tokens) / len(tokens)
    ttr_log = len(unique_tokens) / math.log(len(tokens) + 1)
    
    word_counts = {}
    for token in tokens:
        word_counts[token] = word_counts.get(token, 0) + 1
    
    repeated_words = sum(1 for count in word_counts.values() if count > 1)
    repetition_rate = repeated_words / len(unique_tokens) if len(unique_tokens) > 0 else 0.0
    
    unique_words = sum(1 for count in word_counts.values() if count == 1)
    unique_word_ratio = unique_words / len(tokens)
    
    function_words = {
    'the', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
    'of', 'with', 'by', 'from', 'up', 'about', 'into', 'through', 'during',
    'he', 'she', 'it', 'we', 'they', 'him', 'her', 'us', 'them',
    'my', 'your', 'his', 'her', 'its', 'our', 'their',
    'am', 'is', 'are', 'were', 'be', 'been', 'being',
    'have', 'has', 'had', 'do', 'does', 'did',
    'will', 'would', 'shall', 'should', 'may', 'might', 'must', 'can', 'could',
    'this', 'that', 'these', 'those',
    'what', 'which', 'who', 'when', 'where', 'why', 'how'}
    
    content_words = [token for token in tokens if token not in function_words]
    lexical_density = len(content_words) / len(tokens) if len(tokens) > 0 else 0.0
    
    word_lengths = [len(token) for token in tokens]
    word_length_variance = np.var(word_lengths) if len(word_lengths) > 1 else 0.0
    
    return {
        'ttr': ttr,
        'ttr_log': ttr_log,
        'repetition_rate': repetition_rate,
        'unique_word_ratio': unique_word_ratio,
        'lexical_density': lexical_density,
        'word_length_variance': word_length_variance
    }


# =============================================================================
# Loss Functions
# =============================================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.1):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
    
    def forward(self, logits, targets):
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + self.label_smoothing / 2
        
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_weight = (1 - pt) ** self.gamma
        alpha_weight = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        focal_loss = alpha_weight * focal_weight * bce_loss
        
        return focal_loss.mean()


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# =============================================================================
# WavLM + TTR-Enhanced Transformer Model
# =============================================================================
class WavLMTTREnhancedTransformerModel(nn.Module):
    def __init__(
        self,
        d_model=256,
        nhead=8,
        num_encoder_layers=3,
        dim_feedforward=512,
        dropout=0.3,
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM
    ):
        super(WavLMTTREnhancedTransformerModel, self).__init__()
        
        self.d_model = d_model
        
        # Q-type embedding
        self.q_type_embedding = nn.Embedding(q_type_vocab_size, q_type_embed_dim)
        
        # TTR projection
        self.ttr_projection = nn.Sequential(
            nn.Linear(TTR_ENHANCED_DIM, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, 32)
        )
        
        # Input projection (WavLM + Q-type + TTR)
        input_dim = WAVLM_DIM + q_type_embed_dim + 32
        self.input_projection = nn.Linear(input_dim, d_model)
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers
        )
        
        # CLS token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, batch_wavlm, batch_ttr_enhanced, batch_q_type_ids, num_utterances_list):
        batch_size = len(num_utterances_list)
        device = batch_wavlm.device
        
        # Embeddings
        q_type_embs = self.q_type_embedding(batch_q_type_ids)
        ttr_projected = self.ttr_projection(batch_ttr_enhanced)
        
        # Concatenate features
        combined_features = torch.cat([
            batch_wavlm,
            q_type_embs,
            ttr_projected
        ], dim=1)
        
        # Split by participants
        participant_sequences = []
        start_idx = 0
        
        for num_utts in num_utterances_list:
            end_idx = start_idx + num_utts
            participant_sequences.append(combined_features[start_idx:end_idx])
            start_idx = end_idx
        
        # Padding
        max_num_utterances = max(num_utterances_list)
        padded_sequences = []
        attention_masks = []
        
        for seq, num_utts in zip(participant_sequences, num_utterances_list):
            if num_utts < max_num_utterances:
                padding = torch.zeros(
                    max_num_utterances - num_utts,
                    seq.size(1),
                    device=device
                )
                seq = torch.cat([seq, padding], dim=0)
            
            padded_sequences.append(seq)
            
            mask = torch.zeros(max_num_utterances, dtype=torch.bool, device=device)
            mask[num_utts:] = True
            attention_masks.append(mask)
        
        padded_sequences = torch.stack(padded_sequences)
        attention_masks = torch.stack(attention_masks)
        
        # Project to d_model
        x = self.input_projection(padded_sequences)
        
        # Add CLS token
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        # Extend mask for CLS token
        cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=device)
        extended_mask = torch.cat([cls_mask, attention_masks], dim=1)
        
        # Positional encoding
        x = self.pos_encoder(x)
        
        # Transformer encoding
        encoded = self.transformer_encoder(
            x,
            src_key_padding_mask=extended_mask
        )
        
        # CLS output
        cls_output = encoded[:, 0, :]
        logits = self.classifier(cls_output)
        
        # Attention weights (for visualization)
        utterance_features = encoded[:, 1:, :]
        attention_weights = torch.norm(utterance_features, dim=2)
        attention_weights = attention_weights.masked_fill(attention_masks, 0)
        attention_weights = F.softmax(attention_weights + (attention_masks.float() * -1e9), dim=1)
        
        return logits, attention_weights


# =============================================================================
# Dataset & Collate
# =============================================================================
class WavLMUtteranceDataset(Dataset):
    def __init__(self, participant_data):
        self.data = []
        for pid, info in participant_data.items():
            self.data.append({
                'pid': pid,
                'label': info['label'],
                'utterances': info['utterances'],
                'num_utterances': info['num_utterances']
            })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]


def wavlm_collate_fn(batch):
    batch_labels = []
    all_utterances = []
    num_utterances_list = []
    
    for item in batch:
        batch_labels.append(item['label'])
        all_utterances.extend(item['utterances'])
        num_utterances_list.append(item['num_utterances'])
    
    batch_wavlm = []
    batch_ttr_enhanced = []
    batch_q_type_ids = []
    
    for utt in all_utterances:
        batch_wavlm.append(utt['wavlm'])
        batch_q_type_ids.append(utt['q_type_id'])
        
        text = utt.get('text', '')
        ttr_features = extract_enhanced_ttr_features(text)
        
        ttr_vector = [
            ttr_features['ttr'],
            ttr_features['ttr_log'],
            ttr_features['repetition_rate'],
            ttr_features['unique_word_ratio'],
            ttr_features['lexical_density'],
            ttr_features['word_length_variance']
        ]
        batch_ttr_enhanced.append(ttr_vector)
    
    batch_wavlm = torch.FloatTensor(np.array(batch_wavlm))
    batch_ttr_enhanced = torch.FloatTensor(np.array(batch_ttr_enhanced))
    batch_q_type_ids = torch.LongTensor(batch_q_type_ids)
    batch_labels = torch.FloatTensor(batch_labels).unsqueeze(1)
    
    return {
        'batch_wavlm': batch_wavlm,
        'batch_ttr_enhanced': batch_ttr_enhanced,
        'batch_q_type_ids': batch_q_type_ids,
        'batch_labels': batch_labels,
        'num_utterances_list': num_utterances_list
    }


# =============================================================================
# 데이터 로드
# =============================================================================
def load_and_split_wavlm_data():
    print(f"{'='*70}")
    print(f"📂 WavLM 데이터 로드 중... (TTR Enhanced + Optuna)")
    print(f"{'='*70}")
    
    with open(WAVLM_PREPROCESSED_DATA_PATH, 'rb') as f:
        wavlm_dataset = pickle.load(f)
    
    meta_df = pd.read_csv(os.path.join(BASE_PATH, "metadataset.csv"))
    meta_df['Participant_ID'] = meta_df['Participant_ID'].astype(str)
    
    train_pids = meta_df[meta_df['Group'] == 'Train']['Participant_ID'].tolist()
    val_pids = meta_df[meta_df['Group'] == 'Validation']['Participant_ID'].tolist()
    test_pids = meta_df[meta_df['Group'] == 'Test']['Participant_ID'].tolist()
    
    train_pids = [pid for pid in train_pids if pid in wavlm_dataset]
    val_pids = [pid for pid in val_pids if pid in wavlm_dataset]
    test_pids = [pid for pid in test_pids if pid in wavlm_dataset]
    
    train_labels = [wavlm_dataset[pid]['label'] for pid in train_pids]
    val_labels = [wavlm_dataset[pid]['label'] for pid in val_pids]
    test_labels = [wavlm_dataset[pid]['label'] for pid in test_pids]
    
    print(f"총 참가자 수: {len(wavlm_dataset)}")
    print(f"  - 정상 (0): {sum(1 for d in wavlm_dataset.values() if d['label'] == 0)}")
    print(f"  - 우울증 (1): {sum(1 for d in wavlm_dataset.values() if d['label'] == 1)}")
    
    print(f"\n메타데이터 기준 분할:")
    print(f"  - Train: {len(train_pids)} (0: {train_labels.count(0)}, 1: {train_labels.count(1)})")
    print(f"  - Val:   {len(val_pids)} (0: {val_labels.count(0)}, 1: {val_labels.count(1)})")
    print(f"  - Test:  {len(test_pids)} (0: {test_labels.count(0)}, 1: {test_labels.count(1)})")
    
    train_data = {pid: wavlm_dataset[pid] for pid in train_pids}
    val_data = {pid: wavlm_dataset[pid] for pid in val_pids}
    test_data = {pid: wavlm_dataset[pid] for pid in test_pids}
    
    train_dataset = WavLMUtteranceDataset(train_data)
    val_dataset = WavLMUtteranceDataset(val_data)
    test_dataset = WavLMUtteranceDataset(test_data)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                             collate_fn=wavlm_collate_fn, num_workers=0, 
                             pin_memory=True if torch.cuda.is_available() else False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                           collate_fn=wavlm_collate_fn, num_workers=0,
                           pin_memory=True if torch.cuda.is_available() else False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            collate_fn=wavlm_collate_fn, num_workers=0,
                            pin_memory=True if torch.cuda.is_available() else False)
    
    return train_loader, val_loader, test_loader


# =============================================================================
# 평가 - Balanced F1 계산
# =============================================================================
def evaluate_wavlm(model, dataloader, criterion, threshold=0.5):
    """평가 함수 - Balanced F1 계산"""
    model.eval()
    
    all_labels = []
    all_probs = []
    total_loss = 0.0
    
    with torch.no_grad():
        for batch in dataloader:
            batch_wavlm = batch['batch_wavlm'].to(DEVICE)
            batch_ttr_enhanced = batch['batch_ttr_enhanced'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            logits, _ = model(
                batch_wavlm,
                batch_ttr_enhanced,
                batch_q_type_ids,
                batch['num_utterances_list']
            )
            
            loss = criterion(logits, batch_labels)
            total_loss += loss.item()
            
            probs = torch.sigmoid(logits)
            
            all_labels.extend(batch_labels.cpu().numpy().flatten())
            all_probs.extend(probs.cpu().numpy().flatten())
    
    avg_loss = total_loss / len(dataloader)
    
    # Threshold 적용
    all_preds = (np.array(all_probs) > threshold).astype(int)
    
    # 전체 F1
    overall_f1 = f1_score(all_labels, all_preds, average='binary')
    
    # 클래스별 F1
    f1_per_class = f1_score(all_labels, all_preds, average=None)
    f1_normal = f1_per_class[0]
    f1_depression = f1_per_class[1]
    
    # Balanced F1 (조화평균)
    if f1_normal > 0 and f1_depression > 0:
        balanced_f1 = 2 * (f1_normal * f1_depression) / (f1_normal + f1_depression)
    else:
        balanced_f1 = 0.0
    
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    
    # Specificity
    tn = sum((l == 0 and p == 0) for l, p in zip(all_labels, all_preds))
    fp = sum((l == 0 and p == 1) for l, p in zip(all_labels, all_preds))
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    return {
        'loss': avg_loss,
        'overall_f1': overall_f1,
        'balanced_f1': balanced_f1,
        'f1_normal': f1_normal,
        'f1_depression': f1_depression,
        'precision': precision,
        'recall': recall,
        'specificity': specificity,
        'labels': all_labels,
        'probs': all_probs,
        'preds': all_preds
    }


def find_optimal_threshold(labels, probs, metric='balanced_f1'):
    """최적 threshold 찾기"""
    best_score = 0.0
    best_threshold = 0.5
    best_metrics = {
        'threshold': 0.5,
        'balanced_f1': 0.0,
        'overall_f1': 0.0,
        'f1_normal': 0.0,
        'f1_depression': 0.0,
        'precision': 0.0,
        'recall': 0.0
    }
    
    for thresh in np.arange(0.1, 0.9, 0.01):
        preds = (np.array(probs) > thresh).astype(int)
        
        if len(np.unique(preds)) < 2:
            continue
        
        overall_f1 = f1_score(labels, preds, average='binary', zero_division=0)
        f1_per_class = f1_score(labels, preds, average=None, zero_division=0)
        
        if len(f1_per_class) < 2:
            continue
            
        f1_normal = f1_per_class[0]
        f1_depression = f1_per_class[1]
        
        if f1_normal > 0 and f1_depression > 0:
            balanced_f1 = 2 * (f1_normal * f1_depression) / (f1_normal + f1_depression)
        else:
            balanced_f1 = 0.0
        
        recall = recall_score(labels, preds, zero_division=0)
        precision = precision_score(labels, preds, zero_division=0)
        
        if metric == 'balanced_f1':
            score = balanced_f1
        elif metric == 'overall_f1':
            score = overall_f1
        elif metric == 'recall':
            score = recall
        else:
            score = balanced_f1
        
        if score > best_score:
            best_score = score
            best_threshold = thresh
            best_metrics = {
                'threshold': thresh,
                'balanced_f1': balanced_f1,
                'overall_f1': overall_f1,
                'f1_normal': f1_normal,
                'f1_depression': f1_depression,
                'precision': precision,
                'recall': recall
            }
    
    if best_score == 0.0:
        preds = (np.array(probs) > 0.5).astype(int)
        
        if len(np.unique(preds)) >= 2:
            overall_f1 = f1_score(labels, preds, average='binary', zero_division=0)
            f1_per_class = f1_score(labels, preds, average=None, zero_division=0)
            
            if len(f1_per_class) >= 2:
                f1_normal = f1_per_class[0]
                f1_depression = f1_per_class[1]
                
                if f1_normal > 0 and f1_depression > 0:
                    balanced_f1 = 2 * (f1_normal * f1_depression) / (f1_normal + f1_depression)
                else:
                    balanced_f1 = 0.0
                
                best_metrics = {
                    'threshold': 0.5,
                    'balanced_f1': balanced_f1,
                    'overall_f1': overall_f1,
                    'f1_normal': f1_normal,
                    'f1_depression': f1_depression,
                    'precision': precision_score(labels, preds, zero_division=0),
                    'recall': recall_score(labels, preds, zero_division=0)
                }
    
    return best_threshold, best_metrics


# =============================================================================
# Optuna Objective
# =============================================================================
def objective_wavlm(trial, train_loader, val_loader):
    """Optuna objective function - WavLM 버전"""
    # 하이퍼파라미터 샘플링
    d_model = trial.suggest_categorical('d_model', [128, 256, 384])
    nhead = trial.suggest_categorical('nhead', [4, 8])
    num_encoder_layers = trial.suggest_int('num_encoder_layers', 2, 4)
    dim_feedforward = trial.suggest_categorical('dim_feedforward', [256, 512, 1024])
    dropout = trial.suggest_float('dropout', 0.1, 0.5)
    
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 5e-4, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)
    
    focal_alpha = trial.suggest_float('focal_alpha', 0.3, 0.7)
    focal_gamma = trial.suggest_float('focal_gamma', 1.0, 3.0)
    label_smoothing = trial.suggest_float('label_smoothing', 0.0, 0.15)
    
    # 모델 생성
    wavlm_model = WavLMTTREnhancedTransformerModel(
        d_model=d_model,
        nhead=nhead,
        num_encoder_layers=num_encoder_layers,
        dim_feedforward=dim_feedforward,
        dropout=dropout
    ).to(DEVICE)
    
    # Loss & Optimizer
    criterion = FocalLoss(alpha=focal_alpha, gamma=focal_gamma, label_smoothing=label_smoothing)
    optimizer = optim.AdamW(wavlm_model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    # Scheduler
    warmup_epochs = 3
    warmup_scheduler = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
    cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=OPTUNA_EPOCHS - warmup_epochs, eta_min=1e-6)
    scheduler = optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs])
    
    best_balanced_f1 = 0.0
    patience_counter = 0
    max_patience = 8
    
    for epoch in range(OPTUNA_EPOCHS):
        # Training
        wavlm_model.train()
        train_loss = 0.0
        
        for batch in train_loader:
            batch_wavlm = batch['batch_wavlm'].to(DEVICE)
            batch_ttr_enhanced = batch['batch_ttr_enhanced'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            optimizer.zero_grad()
            logits, _ = wavlm_model(
                batch_wavlm,
                batch_ttr_enhanced,
                batch_q_type_ids,
                batch['num_utterances_list']
            )
            
            loss = criterion(logits, batch_labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(wavlm_model.parameters(), GRADIENT_CLIP)
            optimizer.step()
            
            train_loss += loss.item()
        
        scheduler.step()
        
        # Validation
        val_results = evaluate_wavlm(wavlm_model, val_loader, criterion, threshold=0.5)
        
        # Threshold 최적화
        best_threshold, threshold_metrics = find_optimal_threshold(
            val_results['labels'], 
            val_results['probs'], 
            metric='balanced_f1'
        )
        
        balanced_f1 = threshold_metrics['balanced_f1']
        
        # Pruning
        if epoch >= 5:
            trial.report(balanced_f1, epoch)
            
            if trial.should_prune():
                raise optuna.TrialPruned()
        
        # Best model tracking
        if balanced_f1 > best_balanced_f1:
            best_balanced_f1 = balanced_f1
            patience_counter = 0
        else:
            patience_counter += 1
            
            if patience_counter >= max_patience:
                break
    
    return best_balanced_f1


# =============================================================================
# Training with Best Params
# =============================================================================
def train_wavlm_with_best_params(best_params, train_loader, val_loader):
    """최적 하이퍼파라미터로 WavLM 모델 전체 학습"""
    print(f"\n{'='*70}")
    print(f"🚀 최적 하이퍼파라미터로 WavLM 모델 전체 학습 시작")
    print(f"{'='*70}")
    for key, value in best_params.items():
        print(f"  {key}: {value}")
    print(f"{'='*70}\n")
    
    # 모델 생성
    wavlm_model = WavLMTTREnhancedTransformerModel(
        d_model=best_params['d_model'],
        nhead=best_params['nhead'],
        num_encoder_layers=best_params['num_encoder_layers'],
        dim_feedforward=best_params['dim_feedforward'],
        dropout=best_params['dropout']
    ).to(DEVICE)
    
    criterion = FocalLoss(
        alpha=best_params['focal_alpha'],
        gamma=best_params['focal_gamma'],
        label_smoothing=best_params['label_smoothing']
    )
    
    optimizer = optim.AdamW(
        wavlm_model.parameters(),
        lr=best_params['learning_rate'],
        weight_decay=best_params['weight_decay']
    )
    
    warmup_epochs = 5
    warmup_scheduler = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
    cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - warmup_epochs, eta_min=1e-6)
    scheduler = optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs])
    
    best_balanced_f1 = 0.0
    best_threshold = 0.5
    patience_counter = 0
    
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_balanced_f1': [],
        'val_overall_f1': [],
        'val_f1_normal': [],
        'val_f1_depression': [],
        'val_precision': [],
        'val_recall': [],
        'val_specificity': []
    }
    
    for epoch in range(NUM_EPOCHS):
        # Training
        wavlm_model.train()
        train_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
        for batch in pbar:
            batch_wavlm = batch['batch_wavlm'].to(DEVICE)
            batch_ttr_enhanced = batch['batch_ttr_enhanced'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            optimizer.zero_grad()
            logits, _ = wavlm_model(
                batch_wavlm,
                batch_ttr_enhanced,
                batch_q_type_ids,
                batch['num_utterances_list']
            )
            
            loss = criterion(logits, batch_labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(wavlm_model.parameters(), GRADIENT_CLIP)
            optimizer.step()
            
            train_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})
        
        avg_train_loss = train_loss / len(train_loader)
        
        # Validation
        val_results = evaluate_wavlm(wavlm_model, val_loader, criterion, threshold=0.5)
        
        # Threshold 최적화
        opt_threshold, threshold_metrics = find_optimal_threshold(
            val_results['labels'],
            val_results['probs'],
            metric='balanced_f1'
        )
        
        # History 저장
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(val_results['loss'])
        history['val_balanced_f1'].append(threshold_metrics['balanced_f1'])
        history['val_overall_f1'].append(threshold_metrics['overall_f1'])
        history['val_f1_normal'].append(threshold_metrics['f1_normal'])
        history['val_f1_depression'].append(threshold_metrics['f1_depression'])
        history['val_precision'].append(threshold_metrics['precision'])
        history['val_recall'].append(threshold_metrics['recall'])
        
        # Specificity
        opt_preds = (np.array(val_results['probs']) > opt_threshold).astype(int)
        tn = sum((l == 0 and p == 0) for l, p in zip(val_results['labels'], opt_preds))
        fp = sum((l == 0 and p == 1) for l, p in zip(val_results['labels'], opt_preds))
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        history['val_specificity'].append(specificity)
        
        scheduler.step()
        
        # 출력
        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Val Loss: {val_results['loss']:.4f}")
        print(f"  Optimal Threshold: {opt_threshold:.3f}")
        print(f"  Balanced F1: {threshold_metrics['balanced_f1']:.4f} ⭐")
        print(f"  Overall F1: {threshold_metrics['overall_f1']:.4f}")
        print(f"  F1 Normal (0): {threshold_metrics['f1_normal']:.4f}")
        print(f"  F1 Depression (1): {threshold_metrics['f1_depression']:.4f}")
        print(f"  Precision: {threshold_metrics['precision']:.4f}")
        print(f"  Recall: {threshold_metrics['recall']:.4f}")
        print(f"  Specificity: {specificity:.4f}")
        
        # Best model 저장
        if threshold_metrics['balanced_f1'] > best_balanced_f1:
            best_balanced_f1 = threshold_metrics['balanced_f1']
            best_threshold = opt_threshold
            patience_counter = 0
            
            torch.save({
                'epoch': epoch,
                'model_state_dict': wavlm_model.state_dict(),
                'best_params': best_params,
                'balanced_f1': best_balanced_f1,
                'optimal_threshold': best_threshold,
                'history': history,
                'threshold_metrics': threshold_metrics
            }, os.path.join(BASE_PATH, 'best_wavlm_ttr_enhanced_optuna_model.pt'))
            
            print(f"  ✅ Best model saved! (Balanced F1: {best_balanced_f1:.4f})")
        else:
            patience_counter += 1
            print(f"  ⏳ No improvement ({patience_counter}/{EARLY_STOPPING_PATIENCE})")
            
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"\n⚠️  Early stopping!")
                break
        
        print("-" * 70)
    
    return wavlm_model, history, best_threshold


# =============================================================================
# 시각화
# =============================================================================
def plot_training_history(history):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # Loss
    axes[0, 0].plot(history['train_loss'], label='Train Loss')
    axes[0, 0].plot(history['val_loss'], label='Val Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # Balanced F1
    axes[0, 1].plot(history['val_balanced_f1'], label='Balanced F1', color='purple', linewidth=2)
    axes[0, 1].plot(history['val_overall_f1'], label='Overall F1', color='green')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('F1 Score')
    axes[0, 1].set_title('F1 Scores (Balanced vs Overall)')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # Class-wise F1
    axes[0, 2].plot(history['val_f1_normal'], label='F1 Normal (0)', color='blue')
    axes[0, 2].plot(history['val_f1_depression'], label='F1 Depression (1)', color='red')
    axes[0, 2].set_xlabel('Epoch')
    axes[0, 2].set_ylabel('F1 Score')
    axes[0, 2].set_title('Class-wise F1 Scores')
    axes[0, 2].legend()
    axes[0, 2].grid(True)
    
    # Precision
    axes[1, 0].plot(history['val_precision'], label='Precision', color='blue')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Precision')
    axes[1, 0].set_title('Precision')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    # Recall
    axes[1, 1].plot(history['val_recall'], label='Recall', color='red')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Recall')
    axes[1, 1].set_title('Recall')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    # Specificity
    axes[1, 2].plot(history['val_specificity'], label='Specificity', color='orange')
    axes[1, 2].set_xlabel('Epoch')
    axes[1, 2].set_ylabel('Specificity')
    axes[1, 2].set_title('Specificity')
    axes[1, 2].legend()
    axes[1, 2].grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_PATH, 'wavlm_training_history_optuna.png'), dpi=300)
    plt.close()
    print(f"✅ Training history plot saved!")


def plot_confusion_matrix(labels, preds, title="Confusion Matrix"):
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Normal', 'Depression'],
                yticklabels=['Normal', 'Depression'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(title)
    plt.savefig(os.path.join(BASE_PATH, 'wavlm_confusion_matrix_optuna.png'), dpi=300)
    plt.close()
    print(f"✅ Confusion matrix saved!")


def plot_optuna_optimization(study):
    """Optuna 최적화 결과 시각화"""
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Optimization history
    trials = study.trials
    epochs = [trial.number for trial in trials if trial.state == TrialState.COMPLETE]
    values = [trial.value for trial in trials if trial.state == TrialState.COMPLETE]
    
    axes[0].plot(epochs, values, marker='o')
    axes[0].set_xlabel('Trial')
    axes[0].set_ylabel('Balanced F1')
    axes[0].set_title('Optimization History')
    axes[0].grid(True)
    
    # Best value over time
    best_values = []
    current_best = 0
    for val in values:
        current_best = max(current_best, val)
        best_values.append(current_best)
    
    axes[1].plot(epochs, best_values, marker='o', color='green')
    axes[1].set_xlabel('Trial')
    axes[1].set_ylabel('Best Balanced F1')
    axes[1].set_title('Best Value Over Time')
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_PATH, 'wavlm_optuna_optimization.png'), dpi=300)
    plt.close()
    print(f"✅ Optuna optimization plot saved!")


# =============================================================================
# 메인
# =============================================================================
if __name__ == "__main__":
    print(f"\n{'='*70}")
    print(f"🤖 WavLM + TTR Enhanced Transformer + Optuna (Balanced F1 Optimization)")
    print(f"{'='*70}\n")
    
    # 데이터 로드
    train_loader, val_loader, test_loader = load_and_split_wavlm_data()
    
    # Optuna Study 생성
    study = optuna.create_study(
        study_name=STUDY_NAME,
        direction='maximize',
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
    )
    
    print(f"\n{'='*70}")
    print(f"🔍 Optuna 하이퍼파라미터 탐색 시작 (Trials: {N_TRIALS})")
    print(f"{'='*70}\n")
    
    # 최적화 실행
    study.optimize(
        lambda trial: objective_wavlm(trial, train_loader, val_loader),
        n_trials=N_TRIALS,
        show_progress_bar=True
    )
    
    # 최적 결과 출력
    print(f"\n{'='*70}")
    print(f"✅ Optuna 최적화 완료!")
    print(f"{'='*70}")
    print(f"Best Balanced F1: {study.best_value:.4f}")
    print(f"\nBest Hyperparameters:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")
    
    # 최적 파라미터 저장
    with open(os.path.join(BASE_PATH, 'wavlm_best_params_optuna.json'), 'w') as f:
        json.dump(study.best_params, f, indent=2)
    print(f"\n✅ Best parameters saved to wavlm_best_params_optuna.json")
    
    # Optuna 결과 시각화
    plot_optuna_optimization(study)
    
    # 최적 파라미터로 전체 학습
    wavlm_model, history, best_threshold = train_wavlm_with_best_params(
        study.best_params,
        train_loader,
        val_loader
    )
    
    # History 시각화
    plot_training_history(history)
    
    # Test Set 평가
    print(f"\n{'='*70}")
    print(f"📊 Test Set 평가 (최적 WavLM 모델)")
    print(f"{'='*70}")
    
    model_path = os.path.join(BASE_PATH, 'best_wavlm_ttr_enhanced_optuna_model.pt')
    if os.path.exists(model_path):
        checkpoint = torch.load(model_path)
        wavlm_model.load_state_dict(checkpoint['model_state_dict'])
        best_threshold = checkpoint['optimal_threshold']
        
        print(f"💡 최적 threshold: {best_threshold:.3f}")
        
        test_results = evaluate_wavlm(wavlm_model, test_loader, nn.BCEWithLogitsLoss(), threshold=best_threshold)
        
        print(f"\n🎯 Test Set Results:")
        print(f"  Balanced F1: {test_results['balanced_f1']:.4f} ⭐")
        print(f"  Overall F1: {test_results['overall_f1']:.4f}")
        print(f"  F1 Normal (0): {test_results['f1_normal']:.4f}")
        print(f"  F1 Depression (1): {test_results['f1_depression']:.4f}")
        print(f"  Precision: {test_results['precision']:.4f}")
        print(f"  Recall: {test_results['recall']:.4f}")
        print(f"  Specificity: {test_results['specificity']:.4f}")
        
        # Confusion Matrix
        plot_confusion_matrix(test_results['labels'], test_results['preds'], title="Test Set Confusion Matrix (WavLM)")
        
        # Classification Report
        print(f"\n{'='*70}")
        print("분류 보고서:")
        print(f"{'='*70}")
        print(classification_report(
            test_results['labels'],
            test_results['preds'],
            target_names=['Normal', 'Depression'],
            digits=4
        ))
        
        print(f"\n{'='*70}")
        print(f"✅ 모든 과정 완료!")
        print(f"{'='*70}\n")
    else:
        print(f"\n⚠️  모델 파일을 찾을 수 없습니다.")


🤖 WavLM + TTR Enhanced Transformer + Optuna (Balanced F1 Optimization)

📂 WavLM 데이터 로드 중... (TTR Enhanced + Optuna)


[I 2025-12-09 18:07:47,830] A new study created in memory with name: wavlm_ttr_enhanced_balanced_f1


총 참가자 수: 186
  - 정상 (0): 129
  - 우울증 (1): 57

메타데이터 기준 분할:
  - Train: 107 (0: 76, 1: 31)
  - Val:   33 (0: 21, 1: 12)
  - Test:  46 (0: 32, 1: 14)

🔍 Optuna 하이퍼파라미터 탐색 시작 (Trials: 50)



  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-12-09 18:08:03,811] Trial 0 finished with value: 0.7159090909090909 and parameters: {'d_model': 384, 'nhead': 4, 'num_encoder_layers': 3, 'dim_feedforward': 512, 'dropout': 0.48944921723849566, 'learning_rate': 5.9922728337094086e-05, 'weight_decay': 0.00016929140200159034, 'focal_alpha': 0.3982107952040661, 'focal_gamma': 2.2455059207848915, 'label_smoothing': 0.06486035584082783}. Best is trial 0 with value: 0.7159090909090909.
[I 2025-12-09 18:08:19,166] Trial 1 finished with value: 0.39461883408071746 and parameters: {'d_model': 128, 'nhead': 8, 'num_encoder_layers': 2, 'dim_feedforward': 1024, 'dropout': 0.4680282366722778, 'learning_rate': 2.1517436694139336e-05, 'weight_decay': 0.0008056064727318679, 'focal_alpha': 0.6954115414912657, 'focal_gamma': 1.8993190754242606, 'label_smoothing': 0.022089938015337495}. Best is trial 0 with value: 0.7159090909090909.
[I 2025-12-09 18:08:37,519] Trial 2 finished with value: 0.7450980392156862 and parameters: {'d_model': 384, 'nhead

...


Epoch 29/50
  Train Loss: 0.0223
  Val Loss: 0.0727
  Optimal Threshold: 0.250
  Balanced F1: 0.7259 ⭐
  Overall F1: 0.7097
  F1 Normal (0): 0.7429
  F1 Depression (1): 0.7097
  Precision: 0.5789
  Recall: 0.9167
  Specificity: 0.6190
  ⏳ No improvement (13/15)
----------------------------------------------------------------------


Epoch 30/50: 100%|██████████| 14/14 [00:01<00:00, 12.45it/s, loss=0.0512] 



Epoch 30/50
  Train Loss: 0.0267
  Val Loss: 0.0747
  Optimal Threshold: 0.250
  Balanced F1: 0.7216 ⭐
  Overall F1: 0.6897
  F1 Normal (0): 0.7568
  F1 Depression (1): 0.6897
  Precision: 0.5882
  Recall: 0.8333
  Specificity: 0.6667
  ⏳ No improvement (14/15)
----------------------------------------------------------------------


Epoch 31/50: 100%|██████████| 14/14 [00:01<00:00, 12.86it/s, loss=0.0115]



Epoch 31/50
  Train Loss: 0.0226
  Val Loss: 0.0692
  Optimal Threshold: 0.280
  Balanced F1: 0.7549 ⭐
  Overall F1: 0.7333
  F1 Normal (0): 0.7778
  F1 Depression (1): 0.7333
  Precision: 0.6111
  Recall: 0.9167
  Specificity: 0.6667
  ⏳ No improvement (15/15)

⚠️  Early stopping!
✅ Training history plot saved!

📊 Test Set 평가 (최적 WavLM 모델)
💡 최적 threshold: 0.280

🎯 Test Set Results:
  Balanced F1: 0.6543 ⭐
  Overall F1: 0.5946
  F1 Normal (0): 0.7273
  F1 Depression (1): 0.5946
  Precision: 0.4783
  Recall: 0.7857
  Specificity: 0.6250
✅ Confusion matrix saved!

분류 보고서:
              precision    recall  f1-score   support

      Normal     0.8696    0.6250    0.7273        32
  Depression     0.4783    0.7857    0.5946        14

    accuracy                         0.6739        46
   macro avg     0.6739    0.7054    0.6609        46
weighted avg     0.7505    0.6739    0.6869        46


✅ 모든 과정 완료!

